In [1]:
import itertools
from pathlib import Path
import typing

import pandas as pd

from plot_corr_assoc import create_corr_assoc_matrix
from utils import load_data

# Hardcoded paths, change to apply to new data
data_path = Path("data/preprocessed/esa_2018_2021.pkl")
vars_info_path = Path("data/preprocessed/esa_vars_info.pkl")

image_folder = "outputs/corr_assoc_plots"
plot_save_name_template = "corr_assoc_plot_{group}.png"

allow_overwrite = True

# Read in data
data: pd.DataFrame = load_data(data_path)
vars_info: pd.DataFrame = load_data(vars_info_path)

### search for variable information

In [12]:
vars_info[vars_info["name"].str.contains("westost_2")]

,index,name,description,labels,group,type
115,116,westost_2,"Kennung West-/Ost-Deutschland, Berlin extra","{1: 'Westen', 2: 'Osten', 3: 'Berlin'}",socioecon,nominal


### count values of variable

In [11]:
data["westost_2"].value_counts(dropna=False)

westost_2
1    15349
2     2253
3      698
Name: count, dtype: int64

### show how often two variables are equal and different

In [7]:
# Check if two variables are equal (only works if both are continuous or categorical with the same labels)
vari, varj = "hnb30", "typ_hnb"

# True means vari is equal to varj, False means they are different
vars_equal = (data[vari] == data[varj]) | (data[vari].isnull() & data[varj].isnull())
vars_equal.value_counts(normalize=True) # normalize=False for absolute values, normalize=True for percentages

True    1.0
Name: proportion, dtype: float64

In [8]:
# Show instances where they are not equal
data.loc[~vars_equal, (vari, varj)]

,hnb30,typ_hnb


In [9]:
# Check if two categorical variables are the same
vari, varj = "alkkons", "alkrisk"
# Use normalize="index" to show percentages for each index
pd.crosstab(data[vari], data[varj], dropna=False, normalize = False)

alkrisk,1,2,3,6,4,5
alkkons,,,,,,
NaN,0,0,0,0,0,0
0.0,707,0,0,0,0,0
1.0,0,1540,0,0,0,0
2.0,0,0,2590,0,0,0
3.0,0,0,0,11261,0,0
4.0,0,0,0,0,1164,0
5.0,0,0,0,0,0,310
6.0,0,0,0,0,0,44


### create table of highest correlations

In [10]:
# Get variable groups
var_groups = {
    group: vars_info.loc[vars_info["group"] == group, "name"] for group in vars_info["group"].cat.categories
}
var_groups = typing.cast(dict[str, list[str]], var_groups)

# Visualize correlation matrices
treat_cat_as_continuous: list[str] = vars_info.loc[vars_info["type"].isin(["binary", "ordinal"]), "name"].tolist()

corr_assoc_matrix, p_value_matrix = create_corr_assoc_matrix(data, treat_cat_as_continuous)
table = (
    pd.DataFrame(
        [
            [vari, varj, corr_assoc, p_value_matrix.loc[vari, varj]]
            for vari, varj in itertools.combinations(vars_info["name"], r=2)
            if (corr_assoc := corr_assoc_matrix.loc[vari, varj]) > 0.9
        ]
    )
    .set_axis(["vari", "varj", "corr_assoc", "p_value"], axis=1)
    .sort_values("corr_assoc", ascending=False)
)
table

/Users/milena/Desktop/Master/Consulting/esa_consulting/plot_corr_assoc.py:99: SmallSampleWarning: After omitting NaNs, one or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
/Users/milena/Desktop/Master/Consulting/esa_consulting/plot_corr_assoc.py:35: RuntimeWarning: Mean of empty slice
/Users/milena/Desktop/Master/Consulting/esa_consulting/plot_corr_assoc.py:99: SmallSampleWarning: After omitting NaNs, one or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
/Users/milena/Desktop/Master/Consulting/esa_consulting/plot_corr_assoc.py:35: RuntimeWarning: Mean of empty slice
/Users/milena/Desktop/Master/Consulting/esa_consulting/plot_corr_assoc.py:99: SmallSampleWarning: After omitting NaNs, one or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
/Users/milena/Desktop/Master/Consulting/esa

,vari,varj,corr_assoc,p_value
5,Beruf,psecdi_sch,3.618684,1.532700e-05
8,fam,psecdi_sch,3.382035,1.165932e-04
6,Beruf,hnb_fag_sch,2.987963,6.669626e-01
7,erwerb,psecdi_sch,2.919095,1.641132e-02
0,int_art,psecdi_sch,2.907717,1.168662e-01
1,int_art,hnb_fag_sch,2.009933,3.769992e-01
4,staat,hnb_fag_sch,1.999800,4.007832e-01
3,staat,psecdi_sch,1.150046,1.243275e-01
27,hnb30,typ_hnb,1.000000,0.000000e+00
24,ezp30,typ_ezig,1.000000,0.000000e+00


### count the number of different variables

In [6]:
# number of columns 
print(f"Anzahl Variablen in 'data': {len(data.columns)}")

Anzahl Variablen in 'data': 119


### Paare mit (fast) perfekter Assoziation herausfiltern

In [13]:
# Schwelle für "praktisch perfekt assoziiert"
threshold_perfect = 0.9999

perfect_pairs = table.loc[table["corr_assoc"] >= threshold_perfect].copy()
print("Paare mit (nahezu) perfekter Assoziation:")
print(perfect_pairs)

Paare mit (nahezu) perfekter Assoziation:
        vari          varj  corr_assoc       p_value
5      Beruf    psecdi_sch    3.618684  1.532700e-05
8        fam    psecdi_sch    3.382035  1.165932e-04
6      Beruf   hnb_fag_sch    2.987963  6.669626e-01
7     erwerb    psecdi_sch    2.919095  1.641132e-02
0    int_art    psecdi_sch    2.907717  1.168662e-01
1    int_art   hnb_fag_sch    2.009933  3.769992e-01
4      staat   hnb_fag_sch    1.999800  4.007832e-01
3      staat    psecdi_sch    1.150046  1.243275e-01
27     hnb30       typ_hnb    1.000000  0.000000e+00
24     ezp30      typ_ezig    1.000000  0.000000e+00
2         hh       fr12_he    1.000000  0.000000e+00
22    dsm_al       fr12_he    1.000000           NaN
41   fr12_he      dsm_schm    1.000000  3.971862e-24
40   fr12_he        anr_12    1.000000  3.971862e-24
39   fr12_he  Miss_Abh_dro    1.000000           NaN
38   fr12_he   Miss_Abh_am    1.000000           NaN
37   fr12_he        dsm_au    1.000000           NaN
35  

### Automatisch Vorschläge machen, welche Variable gelöscht werden könnte

In [14]:
# Hilfsfunktion: entscheidet, welche Variable wir eher droppen wollen
def is_derived_or_aux_name(var_name: str) -> bool:
    patterns = [
        "fr12_",    # oft Zwischen- oder Hilfsvariablen
        "_30",      # 30-Tage-Kategorien
        "30gr",
        "30kat",
        "kon",      # z.B. alkkon
        "kons",     # z.B. alkkons
    ]
    return any(pat in var_name for pat in patterns)

drop_candidates = []

for _, row in perfect_pairs.iterrows():
    vari = row["vari"]
    varj = row["varj"]
    
    # Wenn eine der beiden Variablen "technischer" aussieht, markiere diese zum Löschen
    if is_derived_or_aux_name(varj) and not is_derived_or_aux_name(vari):
        drop_candidates.append(varj)
    elif is_derived_or_aux_name(vari) and not is_derived_or_aux_name(varj):
        drop_candidates.append(vari)
    else:
        # Falls beide ähnlich "wichtig" aussehen: Droppen wir standardmäßig varj
        drop_candidates.append(varj)

drop_candidates = sorted(set(drop_candidates))

print("\nVorgeschlagene Variablen zum Entfernen (wegen praktisch perfekter Redundanz):")
for v in drop_candidates:
    print(" -", v)


Vorgeschlagene Variablen zum Entfernen (wegen praktisch perfekter Redundanz):
 - alkkons
 - fr12_ao1
 - fr12_cr1
 - fr12_ec1
 - fr12_he
 - fr12_ko1
 - fr12_ls1
 - fr12_meth1
 - fr12_pi1
 - hnb_fag_sch
 - psecdi_sch
 - typ_ezig
 - typ_hnb
